## Positional Encoding

In [2]:
import numpy as np

class PositionalEncoding:
    def __init__(self, d_model, max_sequence_length):
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length
        self.PE = self._create_pe()

    def _create_pe(self):
        position = np.arange(self.max_sequence_length)[:, np.newaxis]
        div_term = np.exp(
            np.arange(0, self.d_model, 2) *
            (-np.log(10000.0) / self.d_model)
        )

        pe = np.zeros((self.max_sequence_length, self.d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        return pe

    def forward(self, x):
        seq_len = x.shape[1]
        return self.PE[:seq_len]

## Layer Normalization

In [3]:
class LayerNormalization:
    def __init__(self, d_model, eps=1e-5):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps

    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.mean((x - mean) ** 2, axis=-1, keepdims=True)
        std = np.sqrt(var + self.eps)
        y = (x - mean) / std
        return self.gamma * y + self.beta

## Scaled Dot Product Attention

In [ ]:
class ScaledDotProductAttention:
    def forward(self, Q, K, V):
        d_k = Q.shape[-1]
        scores = np.matmul(Q, np.transpose(K, (0,1,3,2))) / np.sqrt(d_k)
        attention = self.softmax(scores)
        output = np.matmul(attention, V)
        return output

    def softmax(self, x):
        x = x - np.max(x, axis=-1, keepdims=True)
        exp = np.exp(x)
        return exp / np.sum(exp, axis=-1, keepdims=True)

## Multi-Head Attention

In [5]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_qkv = np.random.randn(d_model, 3 * d_model)
        self.W_o = np.random.randn(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        qkv = np.matmul(x, self.W_qkv)
        qkv = qkv.reshape(batch_size, seq_len,
                          self.num_heads, 3*self.head_dim)
        qkv = np.transpose(qkv, (0,2,1,3))

        Q, K, V = np.split(qkv, 3, axis=-1)

        values = self.attention.forward(Q, K, V)

        values = np.transpose(values, (0,2,1,3))
        values = values.reshape(batch_size, seq_len, self.d_model)

        out = np.matmul(values, self.W_o)
        return out

## Feed Forward

In [6]:
class PositionwiseFeedForward:
    def __init__(self, d_model, hidden):
        self.W1 = np.random.randn(d_model, hidden)
        self.W2 = np.random.randn(hidden, d_model)

    def forward(self, x):
        x = np.matmul(x, self.W1)
        x = np.maximum(0, x)   # ReLU
        x = np.matmul(x, self.W2)
        return x

## Encoder Layer

In [7]:
class EncoderLayer:
    def __init__(self, d_model, ffn_hidden, num_heads):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = LayerNormalization(d_model)
        self.ffn = PositionwiseFeedForward(d_model, ffn_hidden)
        self.norm2 = LayerNormalization(d_model)

    def forward(self, x):
        # Multi-head attention
        residual = x
        x = self.attention.forward(x)
        x = self.norm1.forward(x + residual)

        # Feed Forward
        residual = x
        x = self.ffn.forward(x)
        x = self.norm2.forward(x + residual)

        return x

In [8]:
class Encoder:
    def __init__(self, d_model, ffn_hidden, num_heads, num_layers):
        self.layers = [
            EncoderLayer(d_model, ffn_hidden, num_heads)
            for _ in range(num_layers)
        ]

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

In [9]:
d_model = 512
num_heads = 8
batch_size = 30
max_sequence_length = 200
ffn_hidden = 2048
num_layers = 5

encoder = Encoder(d_model, ffn_hidden, num_heads, num_layers)

x = np.random.randn(batch_size, max_sequence_length, d_model)

# Add positional encoding
pe = PositionalEncoding(d_model, max_sequence_length)
x = x + pe.forward(x)[np.newaxis, :, :]

out = encoder.forward(x)

print(out.shape)

(30, 200, 512)
